# Profile-Based Content Recommender

## Product goal

Create a personalized **What to Watch Next** playlist for each Netflix profile using the latest completed quarter, with the previous year as a lower-weight preference baseline.

The profile's watched titles are evidence about taste. They are **not** the recommendation catalog. The final playlist should primarily contain unseen titles discovered through external catalogs such as TMDB, with optional provider-availability data from another source.

### Initial product behavior

- Generate 15-20 recommendations per profile.
- Favor recent behavior without forgetting durable preferences.
- Exclude previously watched and explicitly abandoned titles by default.
- Allow continuations, new seasons, franchises, and intentional rewatches in separate playlist groups.
- Explain every recommendation in user-facing language.
- Store each generated playlist as a versioned snapshot.

## End-to-end workflow

```text
ViewingEvent rows
      |
      v
Recent-quarter + yearly baseline windows
      |
      v
Profile preference features
      |
      +-------------------------+
      |                         |
      v                         v
Local Title metadata       External catalog discovery
and watched-title set      (TMDB first, optional providers)
      |                         |
      +------------+------------+
                   v
        Candidate normalization/cache
                   |
                   v
       Eligibility and watched filters
                   |
                   v
       Content similarity + behavior score
                   |
                   v
      Diversity and playlist composition
                   |
                   v
       Stored recommendations + reasons
```

The pipeline should be deterministic for a given profile window, catalog snapshot, and algorithm version.

In [ ]:
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta, timezone
from math import exp
from typing import Any, Iterable

import numpy as np
import pandas as pd

ALGORITHM_VERSION = "content-v1"
PLAYLIST_SIZE = 20
QUARTER_WEIGHT = 1.0
YEAR_BASELINE_WEIGHT = 0.25
MIN_METADATA_CONFIDENCE = 0.70
MAX_EXTERNAL_CALLS_PER_PROFILE_REFRESH = 12

## 1. Select the observation windows

Use the latest **completed** calendar quarter so a playlist does not change every day. The trailing year supplies a lower-weight baseline for stable preferences.

An optional product mode could use a rolling 90-day window for more frequent refreshes. Calendar quarters are easier to explain, cache, reproduce, and compare.

In [ ]:
def latest_completed_quarter(as_of: date | None = None) -> tuple[date, date]:
    as_of = as_of or date.today()
    current_quarter_start_month = ((as_of.month - 1) // 3) * 3 + 1
    current_quarter_start = date(as_of.year, current_quarter_start_month, 1)
    quarter_end = current_quarter_start - timedelta(days=1)
    quarter_start_month = ((quarter_end.month - 1) // 3) * 3 + 1
    quarter_start = date(quarter_end.year, quarter_start_month, 1)
    return quarter_start, quarter_end


quarter_start, quarter_end = latest_completed_quarter()
baseline_start = quarter_end - timedelta(days=365)

pd.DataFrame([
    {"window": "primary", "start": quarter_start, "end": quarter_end, "weight": QUARTER_WEIGHT},
    {"window": "baseline", "start": baseline_start, "end": quarter_end, "weight": YEAR_BASELINE_WEIGHT},
])

## 2. Profile signals

| Signal | Derivation | Recommendation use |
|---|---|---|
| Genre affinity | Weighted watch time and completion by genre | Primary content similarity |
| Keyword/theme affinity | TMDB keywords and overview features | Distinguishes titles inside broad genres |
| Movie/series preference | Share of engaged watch time by media type | Shapes playlist composition |
| Language preference | Watch time and completion by original language | Supports foreign-language discovery |
| Origin-country preference | Weighted watch time by production country | Improves K-drama and international matching |
| Release-era preference | Weighted watch time by five-year release interval | Balances classics and recent releases |
| Runtime preference | Median and distribution of completed runtimes | Selects titles that fit viewing habits |
| Cast/creator affinity | Repeated engagement with actors, directors, creators | Generates unseen adjacent content |
| Franchise/collection affinity | Completed titles and series continuation | Supports sequels and related series |
| Completion | Estimated watched/runtime ratio | Strong positive engagement signal |
| Abandonment | Low completion with no return | Candidate penalty or exclusion |
| Rewatch behavior | Repeat sessions after likely completion | Durable preference and comfort-watch signal |
| Binge tendency | Episodes and sessions per day/week | Controls series and season recommendations |
| Recency | Exponential time decay | Prioritizes current taste |
| Novelty tolerance | Variety of genres, countries, languages, and titles | Controls exploration percentage |
| Popularity tolerance | Historic preference for mainstream vs niche titles | Prevents popularity from dominating |
| Rating/certification | Observed rating distribution or profile setting | Eligibility and safety filter |

Do not infer that a title was disliked from a single short event without accounting for episode runtimes, credits, accidental starts, and later sessions.

In [ ]:
def recency_weight(watched_at: pd.Series, window_end: pd.Timestamp, half_life_days: int = 45) -> pd.Series:
    age_days = (window_end - pd.to_datetime(watched_at, utc=True)).dt.total_seconds() / 86_400
    return np.exp(-np.log(2) * age_days.clip(lower=0) / half_life_days)


def add_engagement_features(events: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    result = events.copy()
    result["watch_hours"] = result["duration_seconds"] / 3600
    result["recency_weight"] = recency_weight(result["started_at"], window_end)
    result["window_weight"] = np.where(
        pd.to_datetime(result["started_at"], utc=True).dt.date >= quarter_start,
        QUARTER_WEIGHT,
        YEAR_BASELINE_WEIGHT,
    )
    result["engagement_weight"] = (
        result["watch_hours"] * result["recency_weight"] * result["window_weight"]
    )
    return result


def weighted_distribution(frame: pd.DataFrame, column: str) -> dict[str, float]:
    exploded = frame.explode(column).dropna(subset=[column])
    totals = exploded.groupby(column)["engagement_weight"].sum()
    if totals.empty or totals.sum() == 0:
        return {}
    return (totals / totals.sum()).sort_values(ascending=False).to_dict()

## 3. Build the profile taste vector

The current database already provides `ViewingEvent` plus cached `Title` fields for media type, genres, countries, language, release year, runtime, rating, popularity, and TMDB identity.

Phase two should add or cache:

- TMDB keywords
- Cast and crew IDs
- TV creators and networks
- Movie collections and franchises
- Overview text or a stored text embedding
- Provider availability by region
- Adult-content and certification eligibility fields

Use stable external IDs rather than names wherever possible.

In [ ]:
@dataclass
class ProfileTaste:
    profile_id: str
    period_start: date
    period_end: date
    genres: dict[str, float] = field(default_factory=dict)
    keywords: dict[str, float] = field(default_factory=dict)
    languages: dict[str, float] = field(default_factory=dict)
    countries: dict[str, float] = field(default_factory=dict)
    media_types: dict[str, float] = field(default_factory=dict)
    people: dict[str, float] = field(default_factory=dict)
    collections: dict[str, float] = field(default_factory=dict)
    preferred_runtime_minutes: float | None = None
    novelty_score: float = 0.0
    binge_score: float = 0.0
    watched_tmdb_ids: set[tuple[str, int]] = field(default_factory=set)
    abandoned_tmdb_ids: set[tuple[str, int]] = field(default_factory=set)


def build_profile_taste(profile_id: str, events: pd.DataFrame) -> ProfileTaste:
    weighted = add_engagement_features(events, pd.Timestamp(quarter_end, tz="UTC"))
    return ProfileTaste(
        profile_id=profile_id,
        period_start=quarter_start,
        period_end=quarter_end,
        genres=weighted_distribution(weighted, "genres"),
        keywords=weighted_distribution(weighted, "keywords"),
        languages=weighted_distribution(weighted, "original_language"),
        countries=weighted_distribution(weighted, "origin_countries"),
        media_types=weighted_distribution(weighted, "media_type"),
        preferred_runtime_minutes=float(weighted["runtime_minutes"].median())
        if weighted["runtime_minutes"].notna().any() else None,
        watched_tmdb_ids=set(
            weighted.dropna(subset=["tmdb_id"])[["media_type", "tmdb_id"]]
            .itertuples(index=False, name=None)
        ),
    )

## 4. Discover unseen candidates

Candidate generation should combine several external pools. This is more robust than asking one endpoint for a final recommendation list.

### TMDB-first candidate sources

1. **Similar and recommended titles** for a small set of high-engagement seed titles.
2. **Discover queries** using top genres, languages, origin countries, release ranges, ratings, and media type.
3. **Cast, creator, and director credits** for repeated people affinities.
4. **Collections, franchises, and related seasons** for continuation candidates.
5. **Controlled popular/trending pool** used for fallback and exploration, not as the main score.

### Optional external sources

- A provider-availability API or licensed catalog dataset to confirm the title is currently available in the user's region.
- IMDb datasets for stable title IDs, alternate names, years, title type, runtime, genres, principals, and aggregate ratings.
- Wikidata for supplementary country, language, franchise, and creator relationships where licensing permits.
- A maintained Netflix catalog dataset for provider-specific availability. Treat scraped or community datasets as stale unless refreshed and timestamped.

Provider availability and editorial ratings must be stored with `source`, `region`, and `observed_at` because they change over time.

In [ ]:
@dataclass
class Candidate:
    external_id: int
    media_type: str
    title: str
    genres: set[str] = field(default_factory=set)
    keywords: set[str] = field(default_factory=set)
    language: str = ""
    countries: set[str] = field(default_factory=set)
    people_ids: set[str] = field(default_factory=set)
    collection_ids: set[str] = field(default_factory=set)
    runtime_minutes: int | None = None
    release_year: int | None = None
    certification: str = ""
    popularity: float = 0.0
    vote_average: float = 0.0
    vote_count: int = 0
    metadata_confidence: float = 1.0
    sources: set[str] = field(default_factory=set)
    available_regions: set[str] = field(default_factory=set)


def external_cache_key(source: str, media_type: str, external_id: int) -> str:
    return f"{source}:{media_type}:{external_id}"


def candidate_is_eligible(candidate: Candidate, taste: ProfileTaste, region: str = "US") -> bool:
    identity = (candidate.media_type, candidate.external_id)
    if identity in taste.watched_tmdb_ids or identity in taste.abandoned_tmdb_ids:
        return False
    if candidate.metadata_confidence < MIN_METADATA_CONFIDENCE:
        return False
    if candidate.available_regions and region not in candidate.available_regions:
        return False
    return True

## 5. Minimize external API calls

External APIs should expand the catalog, but recommendation requests should not directly fan out into dozens of live calls.

### Cache strategy

- Store one canonical external catalog row per `(source, media_type, external_id)`.
- Store expensive child data separately: keywords, credits, providers, recommendations, and similar-title edges.
- Assign different refresh periods:
  - Core identity and release metadata: 30-90 days.
  - Credits and keywords: 90+ days.
  - Popularity and ratings: 7-30 days.
  - Provider availability: 1-7 days.
  - Failed lookups: negative-cache for 7-30 days.
- Deduplicate discovery requests across profiles using a normalized request signature.
- Run enrichment in background jobs and serve the most recent complete playlist while refreshing.

### Per-profile refresh budget

| Calls | Purpose |
|---:|---|
| 0-4 | Similar/recommendations for the highest-value uncached seed titles |
| 0-4 | Discover queries for dominant and exploratory taste segments |
| 0-2 | Repeated cast/creator or collection candidates |
| 0-2 | Missing details for finalists only |

Use cached candidates first. Only spend calls where the candidate pool lacks enough eligible, diverse titles.

## 6. Candidate scoring

A content-based score should combine preference similarity, engagement quality, and contextual fit. Popularity is a small confidence/fallback signal rather than the primary driver.

Suggested initial weights:

```text
genre similarity          0.24
keyword/theme similarity  0.20
language/country match    0.12
cast/creator match        0.10
media type/runtime fit    0.10
release-era fit           0.07
seed-title relationship   0.07
quality confidence        0.05
controlled popularity     0.05
```

These weights should eventually be learned from offline evaluation and user feedback rather than treated as permanent.

In [ ]:
def affinity_score(values: Iterable[str], preferences: dict[str, float]) -> float:
    values = set(values)
    return min(1.0, sum(preferences.get(value, 0.0) for value in values))


def runtime_fit(candidate_runtime: int | None, preferred_runtime: float | None) -> float:
    if not candidate_runtime or not preferred_runtime:
        return 0.5
    distance = abs(candidate_runtime - preferred_runtime)
    return float(exp(-distance / max(preferred_runtime, 30)))


def confidence_score(candidate: Candidate) -> float:
    vote_confidence = min(1.0, np.log1p(candidate.vote_count) / np.log1p(10_000))
    rating = max(0.0, min(1.0, candidate.vote_average / 10))
    return 0.6 * vote_confidence + 0.4 * rating


def score_candidate(candidate: Candidate, taste: ProfileTaste) -> tuple[float, dict[str, float]]:
    components = {
        "genre": affinity_score(candidate.genres, taste.genres),
        "keyword": affinity_score(candidate.keywords, taste.keywords),
        "language_country": 0.5 * taste.languages.get(candidate.language, 0.0)
            + 0.5 * affinity_score(candidate.countries, taste.countries),
        "people": affinity_score(candidate.people_ids, taste.people),
        "type_runtime": 0.5 * taste.media_types.get(candidate.media_type, 0.0)
            + 0.5 * runtime_fit(candidate.runtime_minutes, taste.preferred_runtime_minutes),
        "release_era": 0.5,  # Replace with a weighted release-period distribution.
        "seed_relationship": 1.0 if "seed-related" in candidate.sources else 0.0,
        "quality": confidence_score(candidate),
        "popularity": min(1.0, np.log1p(candidate.popularity) / 10),
    }
    weights = {
        "genre": 0.24,
        "keyword": 0.20,
        "language_country": 0.12,
        "people": 0.10,
        "type_runtime": 0.10,
        "release_era": 0.07,
        "seed_relationship": 0.07,
        "quality": 0.05,
        "popularity": 0.05,
    }
    score = sum(components[name] * weight for name, weight in weights.items())
    return score, components

## 7. Diversity and playlist composition

A raw top-score list will often contain one genre, franchise, country, or title type. Re-rank the finalists with constraints.

### Proposed 20-title composition

- 10 high-confidence taste matches
- 3 adjacent genre or theme discoveries
- 2 international or language-expansion picks, when appropriate
- 2 creator, cast, or franchise connections
- 2 shorter or context-specific watches
- 1 deliberate wildcard

### Guardrails

- Cap any one primary genre at 40% of the playlist.
- Cap a franchise or collection at two entries.
- Include both movies and series when both have meaningful profile affinity.
- Avoid near-duplicate recommendations.
- Reserve exploration slots only when metadata confidence is adequate.
- Keep continuations and rewatches visibly separate from unseen recommendations.

In [ ]:
def jaccard(left: set[str], right: set[str]) -> float:
    union = left | right
    return len(left & right) / len(union) if union else 0.0


def candidate_similarity(left: Candidate, right: Candidate) -> float:
    return (
        0.55 * jaccard(left.genres, right.genres)
        + 0.35 * jaccard(left.keywords, right.keywords)
        + 0.10 * float(left.media_type == right.media_type)
    )


def diversify(
    scored: list[tuple[Candidate, float]],
    limit: int = PLAYLIST_SIZE,
    relevance_weight: float = 0.78,
) -> list[tuple[Candidate, float]]:
    remaining = scored.copy()
    selected: list[tuple[Candidate, float]] = []

    while remaining and len(selected) < limit:
        def rerank_value(item: tuple[Candidate, float]) -> float:
            candidate, relevance = item
            redundancy = max(
                (candidate_similarity(candidate, chosen) for chosen, _ in selected),
                default=0.0,
            )
            return relevance_weight * relevance - (1 - relevance_weight) * redundancy

        best = max(remaining, key=rerank_value)
        selected.append(best)
        remaining.remove(best)

    return selected

## 8. Explanations

Store structured contributing signals and render a short explanation from the strongest two or three factors.

Examples:

- "Because you spent more time with Korean romantic dramas this quarter."
- "A new crime series from creators you return to often."
- "Matches your preference for fast, 30-minute comedies."
- "A highly rated pick just outside your usual genres."
- "More from the collection you watched this year."

Avoid claiming emotional intent or dislike. Explanations should describe observable behavior and metadata relationships.

## 9. Proposed persistence model

### `ExternalCatalogTitle`

- `source`, `external_id`, `media_type`
- canonical and original titles
- genres, keywords, language, countries
- release year, runtime, certification
- cast/crew IDs, collection IDs, overview/embedding
- popularity, vote average/count, poster
- metadata confidence and timestamps

### `CatalogAvailability`

- catalog title, provider, region
- availability type and deep link
- source and `observed_at`

### `RecommendationSet`

- profile, period start/end, lookback mode
- algorithm version, catalog snapshot timestamp
- status, generated timestamp, expires/refresh timestamp
- serialized profile feature summary

### `Recommendation`

- recommendation set and catalog title
- rank, score, playlist segment
- explanation and contributing signals
- seen state at generation time
- user feedback: opened, saved, dismissed, watched

Do not force unseen catalog candidates into the existing `Title` table until they become watched or the product deliberately adopts a unified catalog model.

## 10. Background job workflow

1. Resolve the target profile and completed-quarter window.
2. Load viewing events and cached metadata.
3. Enrich only high-impact watched titles missing required preference fields.
4. Build and store the profile taste vector.
5. Load reusable cached candidate pools.
6. Spend the external-call budget on missing seed/discovery pools.
7. Normalize, merge, and cache candidates by external ID.
8. Remove watched, abandoned, unavailable, adult, and low-confidence candidates.
9. Score candidates and store component scores.
10. Apply diversity and playlist-segment constraints.
11. Generate explanations and persist the completed snapshot.
12. Surface progress as stages plus percentage.

Suggested progress allocation:

```text
10% load profile history
25% build taste profile
50% discover candidate titles
70% enrich and filter candidates
85% rank and diversify
100% store playlist
```

## 11. Evaluation

### Offline backtest

Hide the next quarter, build recommendations from prior history, and measure whether subsequently watched titles appear in the recommendation pool.

- Recall@10 and Recall@20
- NDCG@10 and NDCG@20
- Catalog coverage
- Genre/country/language diversity
- Novelty and popularity distribution
- Already-watched leakage
- Availability accuracy by region
- Metadata match confidence

### Online product signals

- Recommendation opened
- Added to personal list
- Dismissed or marked not interested
- Later watched
- Playlist refresh requested

The strongest learning signal is a later watch, but clicks and saves provide faster feedback. Keep explicit feedback profile-specific.

## 12. Delivery plan

### Phase 1: explainable MVP

- Quarter plus yearly baseline profile vector.
- Existing Title metadata plus TMDB genres, language, country, runtime, and release year.
- Unseen candidates from cached TMDB similar/recommendation and discover pools.
- Watched-title exclusion, weighted scoring, diversity re-ranking, and explanations.
- Persisted playlist snapshot and background progress.

### Phase 2: stronger semantic matching

- Keywords, people, collections, networks, and overview embeddings.
- Regional provider availability.
- Abandonment and completion estimation.
- Explicit recommendation feedback.

### Phase 3: learned ranking

- Backtest using future-quarter watches.
- Learn scoring weights from interaction outcomes.
- Calibrate exploration per profile.
- Consider collaborative signals only after privacy, sparsity, and cold-start requirements are defined.

### Open decisions

- Whether the playlist is Netflix-only or provider-agnostic.
- Which region determines availability.
- How to identify a deliberate abandonment versus an incomplete session.
- Whether profile maturity ratings are inferred, user-configured, or inherited from Netflix profile data.
- Refresh cadence: quarterly, monthly, after import, or user-triggered with rate limits.